# 01 — Análise exploratória (EDA)

Exploração **inicial e enxuta** das duas bases, focada no que orienta a limpeza (`02_limpeza_dados.ipynb`)
e a estratégia de linkage:

1. **Base de busca** — qualidade dos campos, marcadores (`SN`, `None`), CEP genérico, tipos de logradouro.
2. **CNEFE Bahia** — volume, duplicação, formato de `COD_SETOR`, domínios de tipo/título.
3. **Cruzamento** — cobertura de município/CEP/bairro e um *baseline* de match exato ingênuo.

**Pré-requisito:** Parquets gerados pela ingestão (`make ingest`). Rodar via `make notebook`.

In [1]:
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)

# Notebook autocontido: lê apenas os Parquets gerados pela ingestão (`make ingest`).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BUSCA_PATH = ROOT / "data" / "parquet" / "base_busca"
CNEFE_PATH = ROOT / "data" / "parquet" / "cnefe_bahia"
for path in (BUSCA_PATH, CNEFE_PATH):
    assert (path / "_SUCCESS").exists(), f"{path} não encontrado: execute `make ingest` antes."

spark = (
    SparkSession.builder.appName("01-analise-exploratoria")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

busca = spark.read.parquet(str(BUSCA_PATH)).orderBy("source_row_number")
cnefe = spark.read.parquet(str(CNEFE_PATH))
spark.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/11 01:29:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.3'

## 1. Base de busca

In [2]:
bp = busca.toPandas()
CAMPOS = [c for c in bp.columns if c.startswith("consulta_")]
print(bp.shape)
bp.head(10)

(107, 8)


,query_id,source_row_number,consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,consulta_complemento,consulta_municipio
0,Q0001,2,RODOVIA BR 101,None,BR101,48030260,SN,290070
1,Q0002,3,RUA RIO BAIANO,SN,CENTRO,46830000,None,290130
2,Q0003,4,FAZENDA BICO,None,None,46130000,SN,290200
3,Q0004,5,DISTRITO 2 DE JULHO,22,POCO CENTRAL,45675000,None,290240
4,Q0005,6,RUA SERGIO BOMFIM,332,BARREIRINHAS,47800020,None,290320
5,Q0006,7,None EVILASIO FIRMATO,28,SOCRATES REZENDE,45860000,None,290630
6,Q0007,8,FAZENDA LAGOA GRANDE,None,None,46380000,SN,290660
7,Q0008,9,CONJUNTO URBES NOVA,2,NOVA PASTORA,48390000,SN,290700
8,Q0009,10,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,1,None,48110000,None,290750
9,Q0010,11,RUA CASSIMIRO DE ABREU,89,None,45822000,None,291072


In [3]:
MARCADORES_SN = {"SN", "S/N", "S N"}

perfil = pd.DataFrame({
    "nulos": bp[CAMPOS].isna().sum(),
    "distintos": bp[CAMPOS].nunique(),
    "marcador_SN": bp[CAMPOS].apply(lambda s: s.str.upper().isin(MARCADORES_SN).sum()),
    "comprimentos": bp[CAMPOS].apply(lambda s: sorted(s.dropna().str.len().unique())[:6]),
})
perfil

,nulos,distintos,marcador_SN,comprimentos
consulta_logradouro,0,106,0,"[5, 6, 8, 10, 12, 13]"
consulta_numero,11,61,25,"[1, 2, 3, 4]"
consulta_bairro,35,58,0,"[4, 5, 6, 7, 8, 9]"
consulta_cep,0,95,0,[8]
consulta_complemento,67,19,11,"[1, 2, 4, 5, 6, 7]"
consulta_municipio,0,78,0,[6]


In [4]:
bp["tipo_logradouro"] = bp["consulta_logradouro"].str.split().str[0]
bp["cep_generico"] = bp["consulta_cep"].str.endswith("000")
bp["rural"] = (
    bp["tipo_logradouro"].isin(["FAZENDA", "SITIO", "DISTRITO", "POVOADO", "COMUNIDADE"])
    | bp["consulta_bairro"].fillna("").str.contains("RURAL")
)

print("CEP genérico (xxxxx000):", bp["cep_generico"].sum(), "de", len(bp))
print("Endereços rurais (heurística):", bp["rural"].sum())
print("Municípios distintos:", bp["consulta_municipio"].nunique())
bp["tipo_logradouro"].value_counts()

CEP genérico (xxxxx000): 89 de 107
Endereços rurais (heurística): 23
Municípios distintos: 78


tipo_logradouro
RUA           65
FAZENDA       12
AVENIDA        7
SITIO          3
DISTRITO       3
TRAVESSA       3
CONJUNTO       3
None           2
RODOVIA        1
CONDOMINIO     1
PONTA          1
10R            1
COMUNIDADE     1
PRACA          1
VIA            1
LOTEAMENTO     1
EIXO           1
Name: count, dtype: int64

In [5]:
# Complementos: quanto carregam informação útil vs. ruído
bp["consulta_complemento"].value_counts(dropna=False).head(20)

consulta_complemento
None               67
SN                 11
CASA                8
COELBA              3
E                   3
PROX                1
QD 92 L 03          1
DISTRITO            1
CAMINHO 9           1
QD 7 LT 16          1
PROX MARIA SAO      1
SEDE                1
KM 87               1
SEM NUMERO          1
QD 09 LOTE 07       1
1 ETAPA             1
RUA B               1
DAS MANGABEIRAS     1
T                   1
A                   1
Name: count, dtype: int64

## 2. CNEFE Bahia

In [6]:
cnefe.printSchema()

total = cnefe.count()
enderecos_unicos = cnefe.select("COD_UNICO_ENDERECO").distinct().count()
print(f"linhas: {total:,} | COD_UNICO_ENDERECO distintos: {enderecos_unicos:,} "
      f"| linhas por endereço: {total / enderecos_unicos:.2f}")

root
 |-- COD_UNICO_ENDERECO: string (nullable = true)
 |-- COD_UF: string (nullable = true)
 |-- COD_MUNICIPIO: string (nullable = true)
 |-- COD_DISTRITO: string (nullable = true)
 |-- COD_SUBDISTRITO: string (nullable = true)
 |-- COD_SETOR: string (nullable = true)
 |-- NUM_QUADRA: string (nullable = true)
 |-- NUM_FACE: string (nullable = true)
 |-- CEP: string (nullable = true)
 |-- DSC_LOCALIDADE: string (nullable = true)
 |-- NOM_TIPO_SEGLOGR: string (nullable = true)
 |-- NOM_TITULO_SEGLOGR: string (nullable = true)
 |-- NOM_SEGLOGR: string (nullable = true)
 |-- NUM_ENDERECO: string (nullable = true)
 |-- DSC_MODIFICADOR: string (nullable = true)
 |-- NOM_COMP_ELEM1: string (nullable = true)
 |-- VAL_COMP_ELEM1: string (nullable = true)
 |-- NOM_COMP_ELEM2: string (nullable = true)
 |-- VAL_COMP_ELEM2: string (nullable = true)
 |-- NOM_COMP_ELEM3: string (nullable = true)
 |-- VAL_COMP_ELEM3: string (nullable = true)
 |-- NOM_COMP_ELEM4: string (nullable = true)
 |-- VAL_COMP

linhas: 9,047,296 | COD_UNICO_ENDERECO distintos: 8,528,506 | linhas por endereço: 1.06


In [7]:
COLS_CHAVE = [
    "COD_SETOR", "CEP", "DSC_LOCALIDADE", "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR",
    "NOM_SEGLOGR", "NUM_ENDERECO", "DSC_MODIFICADOR", "DSC_ESTABELECIMENTO",
]
nulos = cnefe.select([
    F.round(F.avg(F.when(F.col(c).isNull() | (F.trim(c) == ""), 1).otherwise(0)) * 100, 2).alias(c)
    for c in COLS_CHAVE
]).toPandas().T.rename(columns={0: "% nulo/vazio"})
nulos

,% nulo/vazio
COD_SETOR,0.00
CEP,0.00
DSC_LOCALIDADE,0.00
NOM_TIPO_SEGLOGR,0.00
NOM_TITULO_SEGLOGR,90.12
NOM_SEGLOGR,0.00
NUM_ENDERECO,0.00
DSC_MODIFICADOR,47.49
DSC_ESTABELECIMENTO,84.36


In [8]:
# Formato do setor: o resultado esperado tem 15 dígitos (sem o sufixo)
(cnefe.groupBy(F.length("COD_SETOR").alias("tamanho"),
               F.expr("right(COD_SETOR, 1)").alias("ultimo_char"))
      .count().orderBy(F.desc("count")).show())

+-------+-----------+-------+
|tamanho|ultimo_char|  count|
+-------+-----------+-------+
|     16|          P|9047296|
+-------+-----------+-------+



In [9]:
# Domínios que viram dicionário de normalização
cnefe.groupBy("NOM_TIPO_SEGLOGR").count().orderBy(F.desc("count")).show(40, truncate=False)
cnefe.groupBy("NOM_TITULO_SEGLOGR").count().orderBy(F.desc("count")).show(25, truncate=False)

+------------------+-------+
|NOM_TIPO_SEGLOGR  |count  |
+------------------+-------+
|RUA               |4818805|
|ESTRADA           |983394 |
|FAZENDA           |664661 |
|AVENIDA           |651028 |
|TRAVESSA          |475648 |
|POVOADO           |367136 |
|PRACA             |123690 |
|CAMINHO           |120021 |
|RODOVIA           |101908 |
|SITIO             |54481  |
|ALAMEDA           |47406  |
|LOTEAMENTO        |40146  |
|VILA              |32462  |
|ASSENTAMENTO      |31409  |
|COMUNIDADE        |31369  |
|CORREDOR          |29080  |
|1A TRAVESSA DA RUA|29075  |
|BECO              |27721  |
|CONDOMINIO        |27523  |
|REGIAO            |27401  |
|2A TRAVESSA DA RUA|26394  |
|ACESSO            |22590  |
|ENTRADA           |18739  |
|LADEIRA           |17997  |
|VIA               |16706  |
|QUADRA            |16252  |
|3A TRAVESSA DA RUA|11713  |
|CONJUNTO          |10464  |
|AGLOMERADO        |9551   |
|LARGO             |7485   |
|RAMAL             |7468   |
|OUTROS       

+------------------+-------+
|NOM_TITULO_SEGLOGR|count  |
+------------------+-------+
|NULL              |8153512|
|SAO               |199454 |
|SANTA             |94735  |
|DOUTOR            |76676  |
|SANTO             |54438  |
|PROFESSOR         |36766  |
|PADRE             |29036  |
|NOSSA SENHORA     |28926  |
|PRESIDENTE        |27975  |
|DOM               |26214  |
|CORONEL           |24910  |
|MARECHAL          |18018  |
|PROFESSORA        |16272  |
|VEREADOR          |15924  |
|BARAO             |15713  |
|SENHOR            |13778  |
|ENGENHEIRO        |12486  |
|DEPUTADO          |11843  |
|DUQUE             |11341  |
|PRINCESA          |10797  |
|ALMIRANTE         |10498  |
|GOVERNADOR        |9184   |
|GENERAL           |8482   |
|DESEMBARGADOR     |7804   |
|TENENTE           |7246   |
+------------------+-------+
only showing top 25 rows



In [10]:
cnefe.select(
    F.round(F.avg((F.col("NUM_ENDERECO") == "0").cast("int")) * 100, 2).alias("% numero_0"),
    F.round(F.avg((F.col("DSC_MODIFICADOR") == "SN").cast("int")) * 100, 2).alias("% modificador_SN"),
    F.round(F.avg((F.col("NOM_SEGLOGR") == "SEM DENOMINACAO").cast("int")) * 100, 2).alias("% sem_denominacao"),
    F.round(F.avg(F.col("CEP").endswith("000").cast("int")) * 100, 2).alias("% cep_generico"),
).show()

+----------+----------------+-----------------+--------------+
|% numero_0|% modificador_SN|% sem_denominacao|% cep_generico|
+----------+----------------+-----------------+--------------+
|     29.34|           55.88|              1.8|         64.16|
+----------+----------------+-----------------+--------------+



## 3. Cruzamento busca × CNEFE

CNEFE aos municípios da busca (cod_municipio_6).

In [11]:
municipios = sorted(bp["consulta_municipio"].unique())
cnefe_alvo = (
    cnefe.where(F.col("cod_municipio_6").isin(municipios))
    .withColumn(
        "logradouro_completo",
        F.trim(F.regexp_replace(
            F.concat_ws(" ", "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"), r"\s+", " ")),
    )
    .cache()
)
print(f"linhas do CNEFE nos municípios da busca: {cnefe_alvo.count():,}")

mun_cnefe = {r[0] for r in cnefe_alvo.select("cod_municipio_6").distinct().collect()}
print("municípios da busca ausentes no CNEFE:", sorted(set(municipios) - mun_cnefe) or "nenhum")

linhas do CNEFE nos municípios da busca: 3,889,595


municípios da busca ausentes no CNEFE: nenhum


In [12]:
# Cobertura de CEP e de bairro (como DSC_LOCALIDADE) dentro do mesmo município
ceps = {(r[0], r[1]) for r in cnefe_alvo.select("cod_municipio_6", "CEP").distinct().collect()}
localidades = {(r[0], r[1]) for r in cnefe_alvo.select("cod_municipio_6", "DSC_LOCALIDADE").distinct().collect()}

bp["cep_no_cnefe"] = [(m, c) in ceps for m, c in zip(bp["consulta_municipio"], bp["consulta_cep"])]
bp["bairro_eh_localidade"] = [
    (m, b) in localidades for m, b in zip(bp["consulta_municipio"], bp["consulta_bairro"].fillna(""))
]
pd.crosstab(bp["cep_generico"], bp["cep_no_cnefe"], margins=True)

cep_no_cnefe,False,True,All
cep_generico,,,
False,4,14,18
True,6,83,89
All,10,97,107


In [13]:
print("bairros informados:", bp["consulta_bairro"].notna().sum(),
      "| coincidem com DSC_LOCALIDADE:", bp["bairro_eh_localidade"].sum())

bairros informados: 72 | coincidem com DSC_LOCALIDADE: 51


### Cruzamento com joins 

`consulta_logradouro` == `tipo + título + nome` do CNEFE, dentro do município.


In [14]:
logradouros = cnefe_alvo.groupBy("cod_municipio_6", "logradouro_completo").agg(
    F.countDistinct("COD_SETOR").alias("n_setores"),
    F.count("*").alias("n_linhas"),
)
baseline = (
    busca.join(
        logradouros,
        (busca.consulta_municipio == logradouros.cod_municipio_6)
        & (F.upper(F.trim("consulta_logradouro")) == logradouros.logradouro_completo),
        "left",
    )
    .select("query_id", "consulta_logradouro", "consulta_numero", "consulta_municipio",
            "n_setores", "n_linhas")
    .orderBy("query_id")
    .toPandas()
)
encontrados = baseline["n_setores"].notna()
print(f"match exato: {encontrados.sum()} de {len(baseline)} "
      f"| com setor único: {(baseline['n_setores'] == 1).sum()}")
baseline.loc[encontrados, "n_setores"].describe()

match exato: 42 de 107 | com setor único: 9


count    42.000000
mean      5.571429
std       5.764242
min       1.000000
25%       2.000000
50%       3.000000
75%       8.000000
max      22.000000
Name: n_setores, dtype: float64

In [15]:
# Não encontrados no baseline: amostra para orientar a limpeza
baseline.loc[~encontrados, ["query_id", "consulta_logradouro", "consulta_municipio"]].head(30)

,query_id,consulta_logradouro,consulta_municipio
1,Q0002,RUA RIO BAIANO,290130
2,Q0003,FAZENDA BICO,290200
3,Q0004,DISTRITO 2 DE JULHO,290240
4,Q0005,RUA SERGIO BOMFIM,290320
5,Q0006,None EVILASIO FIRMATO,290630
7,Q0008,CONJUNTO URBES NOVA,290700
8,Q0009,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,290750
9,Q0010,RUA CASSIMIRO DE ABREU,291072
11,Q0012,RUA PADRE PEDRO PEREIRA PROJETO II,291160
16,Q0017,FAZENDA PALMEIRAS DO RIACHO,291430


In [16]:
# Em geral, quantos setores um logradouro atravessa nesses municípios?
logradouros.select("n_setores").summary("count", "mean", "50%", "75%", "90%", "max").show()

+-------+------------------+
|summary|         n_setores|
+-------+------------------+
|  count|             89616|
|   mean|1.5897495982860204|
|    50%|                 1|
|    75%|                 2|
|    90%|                 3|
|    max|                98|
+-------+------------------+



## 4. Conclusões (preencher após executar)

- [ ] Formato final do setor (tamanho/sufixo) e regra de saída.
- [ ] Taxa de duplicação do CNEFE → chave de deduplicação.
- [ ] Domínio de tipos/títulos → dicionário de abreviações.
- [ ] Cobertura de município, CEP e bairro/localidade → quais chaves servem de bloqueio.
- [ ] Taxa do baseline exato e nº de setores por logradouro → necessidade do número e dos passes fuzzy.
- [ ] Lista de anomalias da busca a tratar na limpeza (`None`, tipo embutido, typos, complementos sem valor).

In [17]:
spark.stop()